In [1]:
import pandas as pd
from pathlib import Path

In [2]:
# Load parquet after applying speed calculation fix (MECE buckets)
parquet_path = Path("/home/eprashar_solutions_corelogic_com/network-idx/data/features/fcc/broadband_coverage/tract/fcc_coverage_tract_AK_02.parquet")
df = pd.read_parquet(parquet_path)
print(f"Info for the dataframe:\n{df.info()}")
print("="*80)
print(f"First few rows of the dataframe:\n{df.head()}")

<class 'pandas.DataFrame'>
RangeIndex: 177 entries, 0 to 176
Data columns (total 23 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   tract_geoid                     177 non-null    str    
 1   state_fips                      177 non-null    str    
 2   state_usps                      177 non-null    str    
 3   estimated_census_housing_units  177 non-null    int64  
 4   estimated_fcc_units             177 non-null    int64  
 5   copper_speed_02_02_only         177 non-null    float64
 6   copper_speed_10_1_only          177 non-null    float64
 7   copper_speed_25_3_only          177 non-null    float64
 8   copper_speed_100_20_only        177 non-null    float64
 9   copper_speed_250_25_only        177 non-null    float64
 10  copper_speed_1000_100_only      177 non-null    float64
 11  cable_speed_02_02_only          177 non-null    float64
 12  cable_speed_10_1_only           177 non-null   

In [7]:
# Check if the percentages exceed 100%
from network_idx.constants import STATE_USPS_TO_FIPS
techs = ['copper', 'fiber', 'cable']
speed_cols = [col for col in df.columns if "speed" in col]

for state, fips in STATE_USPS_TO_FIPS.items():
    fpath = Path(f"/home/eprashar_solutions_corelogic_com/network-idx/data/features/fcc/broadband_coverage/tract/fcc_coverage_tract_{state}_{fips}.parquet")
    df = pd.read_parquet(fpath)
    for tech in techs:
        tech_cols = [col for col in speed_cols if col.startswith(tech)]
        violations = (df[tech_cols] > 1.0).sum()
        violations = violations[violations > 0]
        print(f"For state: {state}; checking technology: {tech}")
        if violations.empty:
            print(f"All AOK")
        else:
            print(f"{tech}: values > 100% found:\n{violations}\n")
    print("="*80)
        

For state: AL; checking technology: copper
All AOK
For state: AL; checking technology: fiber
All AOK
For state: AL; checking technology: cable
All AOK
For state: AK; checking technology: copper
All AOK
For state: AK; checking technology: fiber
All AOK
For state: AK; checking technology: cable
All AOK
For state: AZ; checking technology: copper
All AOK
For state: AZ; checking technology: fiber
All AOK
For state: AZ; checking technology: cable
All AOK
For state: AR; checking technology: copper
All AOK
For state: AR; checking technology: fiber
All AOK
For state: AR; checking technology: cable
All AOK
For state: CA; checking technology: copper
All AOK
For state: CA; checking technology: fiber
All AOK
For state: CA; checking technology: cable
All AOK
For state: CO; checking technology: copper
All AOK
For state: CO; checking technology: fiber
All AOK
For state: CO; checking technology: cable
All AOK
For state: CT; checking technology: copper
All AOK
For state: CT; checking technology: fiber
A

FileNotFoundError: [Errno 2] No such file or directory: '/home/eprashar_solutions_corelogic_com/network-idx/data/features/fcc/broadband_coverage/tract/fcc_coverage_tract_AS_60.parquet'

In [6]:
from network_idx.constants import STATE_USPS_TO_FIPS

techs = ['copper', 'fiber', 'cable']
buckets_100_20_plus = ['speed_100_20_only', 'speed_250_25_only', 'speed_1000_100_only']

# Collect violating tracts across all states/techs
states = {}
violations = []
for k, v in STATE_USPS_TO_FIPS.items():
    states[k] = v
    if k == "WY":
        break
for state, fips in states.items():
    fpath = Path(f"/home/eprashar_solutions_corelogic_com/network-idx/data/features/fcc/broadband_coverage/tract/fcc_coverage_tract_{state}_{fips}.parquet")
    df = pd.read_parquet(fpath)
    for tech in techs:
        cols = [f"{tech}_{b}" for b in buckets_100_20_plus]
        total = df[cols].sum(axis=1)
        mask = total > 1.0 + 1e-9
        if mask.any():
            bad = df.loc[mask, ["tract_geoid", "state_usps"] + cols].copy()
            bad["technology"] = tech
            bad["bucket_sum"] = total[mask]
            violations.append(bad)

if violations:
    violations_df = pd.concat(violations, ignore_index=True)
    print(f"{len(violations_df)} violating tract-tech rows")
    display(violations_df)
else:
    print("No violations found")

6 violating tract-tech rows


,tract_geoid,state_usps,fiber_speed_100_20_only,fiber_speed_250_25_only,fiber_speed_1000_100_only,technology,bucket_sum
0,08001008549,CO,0.0051,0.7513,0.7994,fiber,1.5558
1,08001008552,CO,0.0002,0.0356,0.9694,fiber,1.0052
2,08001008564,CO,0.0003,0.0424,0.9696,fiber,1.0123
3,46083010400,SD,0.0017,0.0074,1.0000,fiber,1.0091
4,47165020101,TN,0.0000,0.0008,1.0000,fiber,1.0008
5,47165020102,TN,0.0000,0.0007,1.0000,fiber,1.0007


In [5]:
print(states)

{'AL': '01'}


In [5]:
# Checking that speed buckets are cumulative (monotonically decreasing) 
from network_idx.constants import STATE_USPS_TO_FIPS

# Check that each speed bucket is cumulative (monotonically decreasing)
speed_cols = [
    "speed_02_02",
    "speed_10_1",
    "speed_25_3",
    "speed_100_20",
    "speed_250_25",
    "speed_1000_100"
]

# For each row, check that the values are non-increasing
def is_cumulative(row):
    return all(row[speed_cols[i]] >= row[speed_cols[i+1]] for i in range(len(speed_cols)-1))
i=1
total_states = len(STATE_USPS_TO_FIPS)
for state, fips in STATE_USPS_TO_FIPS.items():
    csv_file = Path(f"/home/eprashar_solutions_corelogic_com/network-idx/data/extracted/fcc/broadband_coverage/{state}/bdc_{fips}_fixed_broadband_summary_by_geography_place_J25_31mar2026.csv")
    df = pd.read_csv(csv_file)
    df["cumulative_check"] = df.apply(is_cumulative, axis=1)
    
    # Compute MECE buckets
    df["less_than_100_20"] = df["speed_02_02"] - df["speed_100_20"]
    df["speed_100_20_only"] = df["speed_100_20"] - df["speed_250_25"]
    df["more_than_100_20"] = df["speed_250_25"]

    # Check that the buckets are MECE and exhaustive
    df["mece_sum"] = df["less_than_100_20"] + df["speed_100_20_only"] + df["more_than_100_20"]
    df["mece_check"] = (df["mece_sum"] - df["speed_02_02"]).abs() < 1e-6  # allow for float error 
    i+=1 
    # Print summary
    print(df.head(5))
    #print(f"{i}/{total_states}: for state {state}, MECE sum is {df['mece_sum']}")
    #print(f"{i}/{total_states}: for state {state}, rows where MECE buckets do not sum to speed_02_02:", (~df["mece_check"]).sum())
    #print(f"{i}/{total_states}: for state {state}, rows where speed buckets are not cumulative:", (~df["cumulative_check"]).sum())
    # Optionally, show a few rows for visual inspection
    # print(df[["less_than_100_20", "speed_100_20_only", "more_than_100_20", "mece_sum", "speed_02_02"]].head())
    
    
    #print(df.loc[~df["cumulative_check"], speed_cols + ["geography_id", "technology"]])

    # Optionally, show a few rows for visual inspection
    # print(df[speed_cols].head())


  area_data_type geography_type  geography_id geography_desc  \
0          Total   Census Place        100100         Abanda   
1          Total   Census Place        100100         Abanda   
2          Total   Census Place        100100         Abanda   
3          Total   Census Place        100100         Abanda   
4          Total   Census Place        100100         Abanda   

  geography_desc_full  total_units biz_res       technology  speed_02_02  \
0          Abanda, AL           97       R   Any Technology     1.000000   
1          Abanda, AL           97       B   Any Technology     1.000000   
2          Abanda, AL           97       R        All Wired     0.257732   
3          Abanda, AL           97       B        All Wired     0.257732   
4          Abanda, AL           97       R  Any Terrestrial     0.257732   

   speed_10_1  speed_25_3  speed_100_20  speed_250_25  speed_1000_100  \
0         1.0         1.0           1.0           1.0             0.0   
1         1.

KeyboardInterrupt: 

In [2]:
parquet_file = Path("/home/eprashar_solutions_corelogic_com/network-idx/data/features/fcc/broadband_coverage/block/fcc_coverage_block_AL_01.parquet")
df = pd.read_parquet(parquet_file)
print(df.info())
print(df.head(25))

<class 'pandas.DataFrame'>
RangeIndex: 185976 entries, 0 to 185975
Data columns (total 18 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   block_geoid              185976 non-null  str    
 1   state_fips               185976 non-null  str    
 2   state_usps               185976 non-null  str    
 3   county_geoid             185976 non-null  str    
 4   tract_geoid              185976 non-null  str    
 5   place_geoid              101328 non-null  str    
 6   source                   185976 non-null  str    
 7   census_housing_units     185976 non-null  int64  
 8   estimated_fcc_units      185976 non-null  int64  
 9   copper_speed_100_20      129616 non-null  float64
 10  copper_less_than_100_20  129616 non-null  float64
 11  copper_more_than_100_20  129616 non-null  float64
 12  cable_speed_100_20       129616 non-null  float64
 13  cable_less_than_100_20   129616 non-null  float64
 14  cable_more_than

In [6]:
# Tract level file
tract_parquet = Path("/home/eprashar_solutions_corelogic_com/network-idx/data/features/fcc/broadband_coverage/tract/fcc_coverage_tract_AK_02.parquet")
tract_df = pd.read_parquet(tract_parquet)
print(tract_df.info())
print(tract_df.head(25))

<class 'pandas.DataFrame'>
RangeIndex: 177 entries, 0 to 176
Data columns (total 14 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   tract_geoid                     177 non-null    str    
 1   state_fips                      177 non-null    str    
 2   state_usps                      177 non-null    str    
 3   estimated_census_housing_units  177 non-null    int64  
 4   estimated_fcc_units             177 non-null    int64  
 5   copper_speed_100_20             177 non-null    float64
 6   copper_less_than_100_20         177 non-null    float64
 7   copper_more_than_100_20         177 non-null    float64
 8   cable_speed_100_20              177 non-null    float64
 9   cable_less_than_100_20          177 non-null    float64
 10  cable_more_than_100_20          177 non-null    float64
 11  fiber_speed_100_20              177 non-null    float64
 12  fiber_less_than_100_20          177 non-null   